# 학습결과와해상도

추가 학습 결과를 확인한 뒤, 해상도 변경으로 어떤 작은 물체를 새로 찾거나 놓치는지 직접 집계합니다.

함수명·입출력·API 예시는 제공합니다. **작성 구간의 처리 로직을 직접 구성하세요.**
검증 셀은 수정하지 않습니다. 파일 안에서는 위에서부터 실행합니다.


In [1]:
from pathlib import Path
import os, sys, json, time

ROOT = Path(r"C:\Users\User\.ngv_day4_2026\pm\1b2d65c48c7e")

sys.path.insert(0,str(ROOT))
os.environ['MPLCONFIGDIR']=str(ROOT/'.runtime/matplotlib')
os.environ['YOLO_CONFIG_DIR']=str(ROOT/'.runtime/yolo')
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageOps, ImageDraw
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
torch.set_num_threads(4)
plt.rcParams['font.family']='Malgun Gothic'
plt.rcParams['axes.unicode_minus']=False
from ultralytics import YOLO
dataset=ROOT/'dataset_group20_seed42'
if not (dataset/'annotations.json').exists():
    from speed_project import prepare_data
    dataset,_=prepare_data(seed=42)
records=json.loads((dataset/'annotations.json').read_text(encoding='utf-8'))
run=ROOT/'runs/verified'
WEIGHTS=run/'weights/best.pt'


## 앞에서 구현한 기능 재사용 · 제공 코드
IoU·일대일 매칭은 완성 코드로 제공합니다. 지표는 매칭 결과의 TP·FP·FN으로 계산합니다.
`entries`: 사진별 image/gt/boxes/scores 목록. 박스는 원본 픽셀 xyxy 좌표입니다.


In [2]:
def iou(a,b):
    iw=max(0,min(a[2],b[2])-max(a[0],b[0]))
    ih=max(0,min(a[3],b[3])-max(a[1],b[1]))
    intersection=iw*ih
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    union=area_a+area_b-intersection
    return intersection/union if union>0 else 0.
def match(entry,conf=.25,iou_threshold=.5):
    scores=np.asarray(entry['scores'])
    order=[int(i) for i in np.argsort(-scores,kind='stable') if scores[i]>=conf]
    used=set();hits=[]
    for i in order:
        remaining=[j for j in range(len(entry['gt'])) if j not in used]
        j=max(remaining,key=lambda j:iou(entry['boxes'][i],entry['gt'][j])) if remaining else None
        ok=j is not None and iou(entry['boxes'][i],entry['gt'][j])>=iou_threshold
        if ok:used.add(j)
        hits.append(bool(ok))
    return order,hits,[j for j in range(len(entry['gt'])) if j not in used]
def metrics(entries,conf=.25):
    tp=fp=fn=0
    for e in entries:
        order,hits,miss=match(e,conf)
        tp+=sum(hits);fp+=len(hits)-sum(hits);fn+=len(miss)
    p=tp/max(tp+fp,1);r=tp/max(tp+fn,1)
    return dict(TP=tp,FP=fp,FN=fn,precision=p,recall=r,F1=2*p*r/max(p+r,1e-12))
def collect(weights,dataset,imgsz=320,split='val',limit=None):
    model=YOLO(str(weights))
    records=json.loads((dataset/'annotations.json').read_text())
    paths=sorted((dataset/'images'/split).glob('*.png'))
    if limit is not None:paths=paths[:limit]
    if not paths:raise ValueError('평가 이미지가 없습니다.')
    model.predict(str(paths[0]),imgsz=imgsz,device='cpu',conf=.001,iou=.7,verbose=False)
    entries=[];times=[]
    for p in paths:
        start=time.perf_counter()
        r=model.predict(str(p),imgsz=imgsz,device='cpu',conf=.001,iou=.7,verbose=False)[0]
        times.append((time.perf_counter()-start)*1000)
        entries.append({'image':p.name,'gt':records[p.name]['boxes'],'boxes':r.boxes.xyxy.tolist(),'scores':r.boxes.conf.tolist()})
    return entries,{'mean_ms':float(np.mean(times)),'n':len(paths),'imgsz':imgsz,'split':split}

def draw_entry(entry, selected=None, title='예측'):
    image=Image.open(dataset/'images/val'/entry['image']).convert('RGB')
    draw=ImageDraw.Draw(image)
    for box in entry['gt']: draw.rectangle(box,outline='cyan',width=3)
    selected=range(len(entry['boxes'])) if selected is None else selected
    for i in selected: draw.rectangle(entry['boxes'][i],outline='red',width=2)
    plt.figure(figsize=(5,5));plt.imshow(image);plt.axis('off');plt.title(title+' / 청록: 정답, 빨강: 예측');plt.show()


## 추가 학습 효과 · 제공 실행
5·10·20 epoch 모델을 같은 조건으로 평가합니다. 직접 학습하려면 RETRAIN=True입니다.


In [3]:
RETRAIN=True # true로 바꿔야 학습 시작.

if RETRAIN:
    from speed_project import prepare_data,train
    dataset,_=prepare_data(seed=42);run=train(dataset,epochs=20)
learning_rows=[]
for epoch in [5,10,20]:
    entries,_=collect(run/'checkpoints'/f'epoch{epoch}.pt',dataset,imgsz=320,split='val')
    learning_rows.append({'epoch':epoch,**metrics(entries,.25)})
for row in learning_rows:print(row)
plt.plot([r['epoch'] for r in learning_rows],[r['F1'] for r in learning_rows],'o-');plt.xlabel('Epoch');plt.ylabel('검증 F1');plt.ylim(0,1);plt.show()


New https://pypi.org/project/ultralytics/8.4.174 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.221  Python-3.12.10 torch-2.6.0+cpu CPU (13th Gen Intel Core i5-1340P)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\User\.ngv_day4_2026\pm\1b2d65c48c7e\dataset_group20_seed42\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0, mode=train, model=C:\Users\User\.ngv_day4_2026\pm\1b2d65c48c7e\models\yol

KeyboardInterrupt: 

## ✏️ B06 · 정답 물체별 크기와 검출 여부 집계

사진별 예측을 매칭하고 **각 정답 박스마다** 결과 행을 만드세요.
작은 물체는 원본 사진 면적 대비 정답 박스 면적이 small_ratio 미만인 경우입니다. 검출 여부는 같은 정답이 매칭됐는지로 판단합니다.
반환 행 키: image, gt_index, small(bool), detected(bool). 이 수업의 작은 물체 기준이며 COCO 정의와 다릅니다.

### 사용할 수 있는 API
`match(entry,conf)`의 세 번째 반환값은 놓친 정답 번호입니다.
`records[파일명]["width"]`, `["height"]`는 원본 사진 크기입니다.
`enumerate(items)`로 정답 번호와 박스를 함께 가져올 수 있습니다.


### 처리 단계 힌트
1. 사진별 매칭에서 놓친 정답 번호를 가져옵니다.
2. 모든 정답 박스를 순회합니다.
3. 박스 면적/원본 사진 면적으로 크기를 판정합니다.
4. 정답 번호가 놓친 목록에 있는지로 검출 여부를 기록합니다.

**작은 예시** · 원본 100×100에서 5×5 정답 박스의 면적 비율은 0.0025입니다. 기준 0.01보다 작으면 `small=True`입니다.


In [ ]:
# ===== 실습 B06 시작 =====

def object_outcomes(entries, records, small_ratio=.01, conf=.25):
    """
    각 이미지의 Ground Truth(정답) 오브젝트별로 크기(small 여부)와 
    모델 탐지 여부(detected)를 매칭하여 데이터 프레임용 행(rows) 목록을 반환합니다.
    """
    rows = []
    
    for e in entries:
        # 1. 현재 entry에 대한 모델 예측 매칭 결과(놓친 index 목록 등) 추출
        miss = match(e, conf)
        
        # 2. 이미지 ID를 기반으로 records에서 해당 이미지의 너비와 높이를 가져와 면적 계산
        image_id = e['image']
        image_area = records[image_id]['width'] * records[image_id]['height']
        
        # 3. Ground Truth(gt)의 인덱스(j)와 좌표(x1, y1, x2, y2)를 순회
        for j, (x1, y1, x2, y2) in enumerate(e['gt']):
            # 오브젝트가 이미지에서 차지하는 비율 계산
            ratio = (x2 - x1) * (y2 - y1) / image_area
            
            # 결과 row 추가
            rows.append({
                'image': image_id,
                'gt_index': j,
                'small': bool(ratio < small_ratio),
                'detected': bool(j not in miss)  # 명확하게 bool 타입으로 보장
            })
            
    return rows

# ===== 실습 B06 끝 =====


**B06 검증** · 아래 셀이 통과한 뒤 실제 데이터에 적용합니다.


In [ ]:
toy=[{'image':'a','gt':[[0,0,5,5],[20,20,70,70]],'boxes':[[20,20,70,70]],'scores':[.9]}]
sizes={'a':{'width':100,'height':100}}
r=object_outcomes(toy,sizes)
assert r==[{'image':'a','gt_index':0,'small':True,'detected':False},{'image':'a','gt_index':1,'small':False,'detected':True}],'B06: 큰 물체 검출을 작은 물체 검출로 세지 마세요.'
assert object_outcomes([{'image':'a','gt':[],'boxes':[],'scores':[]}],sizes)==[]
edge=[{'image':'a','gt':[[0,0,10,10]],'boxes':[[0,0,10,10]],'scores':[.1]}]
assert object_outcomes(edge,sizes)[0]=={'image':'a','gt_index':0,'small':False,'detected':False}
assert object_outcomes(edge,{'a':{'width':200,'height':100}},conf=0.)[0]['small']
print('B06 통과: 크기 기준·원본 면적·검출 여부·빈 정답')


AssertionError: B06: 큰 물체 검출을 작은 물체 검출로 세지 마세요.

## ✏️ B07 · 해상도 전후 같은 물체 비교

B06의 두 결과 목록을 비교합니다. 목록 순서는 달라도 됩니다. `(image, gt_index)`로 **같은 작은 정답 물체**를 연결하세요.
반환: gained(새로 검출), lost(새로 놓침), still_missed(계속 놓침)의 키 목록. 각 목록은 정렬합니다.
두 결과의 작은 정답 물체 집합이 다르면 ValueError를 발생시킵니다.

### 사용할 수 있는 API
튜플을 사전 키로 사용할 수 있습니다: `mapping[("photo.png",2)]=value`.
`set(mapping)`은 키 집합입니다. `sorted`, 불리언 비교, `raise ValueError("설명")`를 사용할 수 있습니다.


### 처리 단계 힌트
1. 작은 정답 물체만 골라 `(image,gt_index)`를 키로 만듭니다.
2. 두 결과의 키 집합이 같은지 확인합니다.
3. 같은 키의 검출 여부를 비교합니다.
4. 변화 종류별 키 목록을 정렬해 반환합니다.

**작은 예시** · 같은 키의 검출이 `False→True`이면 gained, `True→False`이면 lost입니다. `True→True`는 반환 목록에 넣지 않습니다.


In [ ]:
# ===== 정답 B07 시작 =====

def compare_outcomes(before, after):
    a={(r['image'],r['gt_index']):r for r in before if r['small']}
    b={(r['image'],r['gt_index']):r for r in after if r['small']}
    result={'gained':[],'lost':[],'still_missed':[]}
    for key in sorted(a):
        old,new=a[key]['detected'],b[key]['detected']
        if not old and new:result['gained'].append(key)
        elif old and not new:result['lost'].append(key)
        elif not old and not new:result['still_missed'].append(key)
    return result

# ===== 정답 B07 끝 =====


**B07 검증** · 아래 셀이 통과한 뒤 실제 데이터에 적용합니다.


In [ ]:
before=[dict(image='a',gt_index=i,small=True,detected=v) for i,v in enumerate([False,True,False,True])]
after=[dict(image='a',gt_index=i,small=True,detected=v) for i,v in enumerate([True,False,False,True])][::-1]
assert compare_outcomes(before,after)=={'gained':[('a',0)],'lost':[('a',1)],'still_missed':[('a',2)]},'B07: 목록 위치가 아니라 같은 정답을 연결하세요.'
assert compare_outcomes([],[])=={'gained':[],'lost':[],'still_missed':[]}
try:compare_outcomes(before,after[:-1])
except ValueError:pass
else:raise AssertionError('B07: 비교 대상이 다르면 오류를 알려야 합니다.')
print('B07 통과: 순서 독립·검출 증가/감소·계속 놓침·대상 일치')


KeyError: ('a', 0)

## 같은 모델의 해상도 비교 · 제공 실행
모델은 20 epoch로 고정합니다. 추론·시간 측정·그래프는 제공하며 작성한 정답별 집계 함수를 사용합니다.
시간은 워밍업 후 같은 20장×3회의 장당 시간 중앙값입니다. 파일 읽기·전후처리를 포함하고 모델 로딩·화면 표시는 제외합니다.


In [ ]:
detector=YOLO(str(run/'checkpoints/epoch20.pt'))
paths=sorted((dataset/'images/val').glob('*.png'))
def infer_one(path,size):
    r=detector.predict(str(path),imgsz=size,conf=.001,iou=.7,device='cpu',verbose=False)[0]
    return dict(image=path.name,gt=records[path.name]['boxes'],boxes=r.boxes.xyxy.tolist(),scores=r.boxes.conf.tolist())
def measure_time(paths,size):
    infer_one(paths[0],size);samples=[]
    for _ in range(3):
        start=time.perf_counter()
        for p in paths:infer_one(p,size)
        samples.append((time.perf_counter()-start)*1000/len(paths))
    return float(np.median(samples))
SIZES=[160,320,640]
all_entries={};all_outcomes={};rows=[]
for size in SIZES:
    entries=[infer_one(p,size) for p in paths];all_entries[size]=entries
    outcomes=object_outcomes(entries,records);all_outcomes[size]=outcomes
    small=[r for r in outcomes if r['small']];found=sum(r['detected'] for r in small)
    row=dict(size=size,small_found=found,small_total=len(small),small_recall=found/len(small) if small else 0.,ms=measure_time(paths[:20],size),**metrics(entries))
    rows.append(row);print(row)
fig,axes=plt.subplots(1,2,figsize=(10,3))
axes[0].plot(SIZES,[r['small_recall'] for r in rows],'o-',label='작은 물체 재현율');axes[0].plot(SIZES,[r['F1'] for r in rows],'o-',label='전체 F1');axes[0].set_ylim(0,1);axes[0].legend()
axes[1].plot(SIZES,[r['ms'] for r in rows],'o-');axes[1].set_ylabel('장당 시간 중앙값(ms)')
for ax in axes:ax.set_xlabel('추론 입력 크기')
plt.show()
changes=compare_outcomes(all_outcomes[SIZES[0]],all_outcomes[SIZES[-1]])
print('160 → 640 작은 물체 변화:',{k:len(v) for k,v in changes.items()})
for category in ['gained','lost','still_missed']:
    if not changes[category]:continue
    filename,gt_index=changes[category][0]
    for size in [SIZES[0],SIZES[-1]]:
        e=next(e for e in all_entries[size] if e['image']==filename)
        draw_entry(e,match(e)[0],f'{category} / 입력 {size} / 정답 번호 {gt_index}')


## 검증으로 설정 확정 · 제공 코드
20 epoch 모델에서 해상도·점수 기준을 함께 선택합니다. 목표 재현율 이상에서 정밀도를 우선하고, 동률이면 재현율·작은 해상도 순서입니다.

In [ ]:
# 학습량은 20 epoch, NMS는 0.7로 고정. 해상도·점수 기준은 검증 사진으로만 선택합니다.
THRESHOLDS=[.15,.4,.7]
MIN_RECALL=.93
candidates=[]
for size in SIZES:
    for threshold in THRESHOLDS:
        candidates.append(dict(size=size,threshold=threshold,**metrics(all_entries[size],threshold)))
eligible=[r for r in candidates if r['recall']>=MIN_RECALL]
selected=max(eligible,key=lambda r:(r['precision'],r['recall'],-r['size'])) if eligible else None
print('검증으로 확정한 설정:',selected)


## 최종 시험 확인 · 선택 실행
실험을 마친 뒤 `RUN_FINAL_TEST=True`로 실행하세요. 시험 결과를 보고 설정을 다시 고르지 않습니다.

In [ ]:
RUN_FINAL_TEST=False
if not RUN_FINAL_TEST:
    print('시험 평가 대기: 실험을 마친 뒤 RUN_FINAL_TEST=True로 실행하세요.')
elif selected is None:
    print('검증에서 목표 재현율을 만족하는 후보가 없어 시험 평가를 건너뜁니다.')
else:
    test_entries,_=collect(run/'checkpoints/epoch20.pt',dataset,imgsz=selected['size'],split='test')
    test_metrics=metrics(test_entries,selected['threshold'])
    print('최종 시험 성능 / 설정 재선택 금지:',test_metrics)
    print('시험 사진 수:',len(test_entries),'정답 물체 수:',sum(len(e['gt']) for e in test_entries))